In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)

# 1. Synthetic diabetes onset dataset (~500 clinical records, 8 features)
n_pat = 500
X_clin = np.random.randn(n_pat, 8)
logits = 1.4 * X_clin[:, 0] + 1.1 * X_clin[:, 1] + 0.7 * X_clin[:, 2] - 0.5 * X_clin[:, 3] - 0.4
probs = 1.0 / (1.0 + np.exp(-logits))
y_diab = (np.random.rand(n_pat) < probs).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X_clin, y_diab, test_size=0.25, random_state=42, stratify=y_diab)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

def make_model(l2_reg=0.0, drop_rate=0.0):
    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None
    m = keras.Sequential([
        layers.Input(shape=(8,)),
        layers.Dense(256, activation="relu", kernel_regularizer=reg),
        layers.Dropout(drop_rate),
        layers.Dense(128, activation="relu", kernel_regularizer=reg),
        layers.Dropout(drop_rate),
        layers.Dense(64, activation="relu", kernel_regularizer=reg),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

records = []

# Baseline (large MLP prone to overfitting)
tf.random.set_seed(42)
base_model = make_model(l2_reg=0.0, drop_rate=0.0)
base_hist = base_model.fit(X_tr, y_tr, validation_split=0.2, epochs=120, batch_size=32, verbose=0)
tr_acc_b = base_model.evaluate(X_tr, y_tr, verbose=0)[1]
te_acc_b = base_model.evaluate(X_te, y_te, verbose=0)[1]
records.append({"Model": "Baseline (Large MLP)", "Test Acc": round(te_acc_b, 4), "Train-Test Gap": round(tr_acc_b - te_acc_b, 4)})

# L2 regularisation at three lambda values
for lam in [1e-4, 1e-3, 1e-2]:
    tf.random.set_seed(42)
    m = make_model(l2_reg=lam, drop_rate=0.0)
    m.fit(X_tr, y_tr, validation_split=0.2, epochs=120, batch_size=32, verbose=0)
    tr_a = m.evaluate(X_tr, y_tr, verbose=0)[1]
    te_a = m.evaluate(X_te, y_te, verbose=0)[1]
    records.append({"Model": f"L2 (λ={lam})", "Test Acc": round(te_a, 4), "Train-Test Gap": round(tr_a - te_a, 4)})

# Dropout at 0.2, 0.3, 0.5
drop_histories = {}
for dr in [0.2, 0.3, 0.5]:
    tf.random.set_seed(42)
    m = make_model(l2_reg=0.0, drop_rate=dr)
    h = m.fit(X_tr, y_tr, validation_split=0.2, epochs=120, batch_size=32, verbose=0)
    drop_histories[dr] = h.history["val_loss"]
    tr_a = m.evaluate(X_tr, y_tr, verbose=0)[1]
    te_a = m.evaluate(X_te, y_te, verbose=0)[1]
    records.append({"Model": f"Dropout ({dr})", "Test Acc": round(te_a, 4), "Train-Test Gap": round(tr_a - te_a, 4)})

# Early Stopping using validation loss
tf.random.set_seed(42)
es_model = make_model(l2_reg=1e-3, drop_rate=0.2)
es_cb = keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
es_hist = es_model.fit(X_tr, y_tr, validation_split=0.2, epochs=120, batch_size=32, callbacks=[es_cb], verbose=0)
tr_a_es = es_model.evaluate(X_tr, y_tr, verbose=0)[1]
te_a_es = es_model.evaluate(X_te, y_te, verbose=0)[1]
records.append({"Model": f"EarlyStopping (ep {len(es_hist.history['loss'])})", "Test Acc": round(te_a_es, 4), "Train-Test Gap": round(tr_a_es - te_a_es, 4)})

df_reg = pd.DataFrame(records)
print("\nRegularisation Comparison Table:")
print(df_reg.to_string(index=False))

# Plot baseline overfitting & validation loss curves
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

axes[0].plot(base_hist.history["loss"], label="Baseline Train Loss")
axes[0].plot(base_hist.history["val_loss"], label="Baseline Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Baseline Overfitting Curve")
axes[0].legend()

axes[1].plot(base_hist.history["val_loss"], "--", label="Baseline Val Loss")
for dr, vloss in drop_histories.items():
    axes[1].plot(vloss, label=f"Dropout {dr}")
axes[1].plot(es_hist.history["val_loss"], lw=2, color="black", label="EarlyStopping")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Loss")
axes[1].set_title("Validation Loss Across Regularisation Methods")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("exp10_regularisation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()